In this notebook, I will use the data generated in poseRecognition.ipynb to extract additional features. The primary focus will be on calculating the angles between joints and measuring the speed of movement.

In [8]:
pip install pandas

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ------- -------------------------------- 1.8/9.7 MB 12.6 MB/s eta 0:00:01
   --------------------- ------------------ 5.2/9.7 MB 13.9 MB/s eta 0:00:01
   ----------------------------------- ---- 8.7/9.7 MB 14.9 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 14.5 MB/s  0:00:00

   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   ---------------------------------------- 0/2 [tzdata]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas]
   -------------------- ------------------- 1/2 [pandas

In [9]:
import pandas as pd


1. Load the existing data

In [23]:
data = pd.read_csv('output\poseDatasied1.csv')
data 

<>:1: SyntaxWarning: invalid escape sequence '\p'
<>:1: SyntaxWarning: invalid escape sequence '\p'
C:\Users\David\AppData\Local\Temp\ipykernel_22296\1618844098.py:1: SyntaxWarning: invalid escape sequence '\p'
  data = pd.read_csv('output\poseDatasied1.csv')


,frame,timestamp,joint,x,y,z
0,0,2.543676,0,0.599596,0.163613,-0.002770
1,0,2.543676,1,0.604323,0.142790,-0.027294
2,0,2.543676,2,0.605793,0.142282,-0.027366
3,0,2.543676,3,0.607432,0.141890,-0.027390
4,0,2.543676,4,0.603356,0.142708,0.010849
...,...,...,...,...,...,...
3196,98,6.094092,28,0.571952,0.855607,0.191839
3197,98,6.094092,29,0.591863,0.896999,-0.031359
3198,98,6.094092,30,0.584016,0.879280,0.194868
3199,98,6.094092,31,0.538105,0.907610,-0.050237


1.2 Data cleanup 

Since the data was generated using MediaPipe, some joints are not relevant for my analysis. The joints are organized as follows:
| ID  | Landmark           |
|-----|------------------|
| 0   | NOSE              |
| 1   | LEFT_EYE_INNER    |
| 2   | LEFT_EYE          |
| 3   | LEFT_EYE_OUTER    |
| 4   | RIGHT_EYE_INNER   |
| 5   | RIGHT_EYE         |
| 6   | RIGHT_EYE_OUTER   |
| 7   | LEFT_EAR          |
| 8   | RIGHT_EAR         |
| 9   | MOUTH_LEFT        |
| 10  | MOUTH_RIGHT       |
| 11  | LEFT_SHOULDER     |
| 12  | RIGHT_SHOULDER    |
| 13  | LEFT_ELBOW        |
| 14  | RIGHT_ELBOW       |
| 15  | LEFT_WRIST        |
| 16  | RIGHT_WRIST       |
| 17  | LEFT_PINKY        |
| 18  | RIGHT_PINKY       |
| 19  | LEFT_INDEX        |
| 20  | RIGHT_INDEX       |
| 21  | LEFT_THUMB        |
| 22  | RIGHT_THUMB       |
| 23  | LEFT_HIP          |
| 24  | RIGHT_HIP         |
| 25  | LEFT_KNEE         |
| 26  | RIGHT_KNEE        |
| 27  | LEFT_ANKLE        |
| 28  | RIGHT_ANKLE       |
| 29  | LEFT_HEEL         |
| 30  | RIGHT_HEEL        |
| 31  | LEFT_FOOT_INDEX   |
| 32  | RIGHT_FOOT_INDEX  |


I will exclude irrelevant joints, such as ears ore pinkys, from the analysis.


In [31]:
needed = (11,12,13,14,15,16,23,24,25,26,27,28,29,30,31,32)


In [35]:
data = data[data["joint"].isin(needed)]
data


,frame,timestamp,joint,x,y,z
11,0,2.543676,11,0.637776,0.228370,-0.196496
12,0,2.543676,12,0.641305,0.242726,0.193960
13,0,2.543676,13,0.565968,0.301557,-0.211803
14,0,2.543676,14,0.584059,0.320715,0.224480
15,0,2.543676,15,0.524369,0.310284,-0.079088
...,...,...,...,...,...,...
3196,98,6.094092,28,0.571952,0.855607,0.191839
3197,98,6.094092,29,0.591863,0.896999,-0.031359
3198,98,6.094092,30,0.584016,0.879280,0.194868
3199,98,6.094092,31,0.538105,0.907610,-0.050237


2. Compute joint angles


In [15]:
import numpy as np

Pivot data to have joints as columns per frame
Columns will be like: joint0_x, joint0_y, joint0_z, joint1_x, ...

In [16]:

pose_pivot = data.pivot(index='frame', columns='joint', values=['x','y','z'])
pose_pivot.columns = [f"{axis}{joint}" for axis, joint in pose_pivot.columns]


Calculate the angle at point b formed by points a-b-c in 3D.
Returns angle in degrees.


In [17]:
def calculate_angle(a, b, c):

    ba = np.array(a) - np.array(b)
    bc = np.array(c) - np.array(b)
    cos_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    cos_angle = np.clip(cos_angle, -1.0, 1.0)  # Numerical stability
    angle = np.arccos(cos_angle)
    return np.degrees(angle)


compute elbow angle (shoulder-elbow-wrist)

In [ ]:
angles = []
for idx, row in pose_pivot.iterrows():
    shoulder = (row['x0'], row['y0'], row['z0'])  
    elbow = (row['x1'], row['y1'], row['z1'])
    wrist = (row['x2'], row['y2'], row['z2'])
    angle = calculate_angle(shoulder, elbow, wrist)
    angles.append(angle)

pose_pivot['elbow_angle'] = angles

pose_pivot


,x0,x1,x2,x3,x4,x5,x6,x7,x8,x9,...,z24,z25,z26,z27,z28,z29,z30,z31,z32,elbow_angle
frame,,,,,,,,,,,,,,,,,,,,,
0,0.599596,0.604323,0.605793,0.607432,0.603356,0.604020,0.604883,0.623072,0.620281,0.604261,...,0.122710,-0.126037,0.135141,-0.094091,0.166577,-0.094126,0.166893,-0.180576,0.119644,112.421143
1,0.598046,0.602347,0.603889,0.605677,0.601723,0.602716,0.603984,0.621228,0.619385,0.604214,...,0.120991,-0.102255,0.143286,-0.063738,0.192298,-0.062081,0.195303,-0.136353,0.150872,113.738871
2,0.597423,0.601649,0.603186,0.604982,0.601254,0.602394,0.603815,0.619995,0.618913,0.604187,...,0.120949,-0.085502,0.156246,-0.033563,0.220653,-0.030661,0.225270,-0.107148,0.175814,114.041718
3,0.597417,0.601511,0.603113,0.604950,0.600985,0.602139,0.603598,0.620005,0.618637,0.604426,...,0.121326,-0.090134,0.163472,-0.042556,0.225840,-0.040340,0.230409,-0.117578,0.184804,114.793348
4,0.597436,0.601532,0.603149,0.604989,0.600981,0.602141,0.603600,0.620074,0.618651,0.604902,...,0.122042,-0.073772,0.171884,-0.008788,0.248966,-0.004759,0.255036,-0.081320,0.204945,116.477897
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,0.456170,0.450662,0.450885,0.451057,0.450711,0.451002,0.451381,0.458470,0.459741,0.465574,...,0.108235,-0.088040,0.128038,-0.042616,0.163751,-0.036814,0.165738,-0.045904,0.155753,121.199292
95,0.456302,0.450802,0.451043,0.451227,0.450793,0.451070,0.451430,0.458743,0.459815,0.465826,...,0.108206,-0.088528,0.128429,-0.043209,0.170044,-0.037599,0.172517,-0.049177,0.162245,120.834280
96,0.456435,0.450972,0.451235,0.451435,0.450907,0.451175,0.451518,0.459023,0.459918,0.466085,...,0.108271,-0.089601,0.139565,-0.043809,0.187129,-0.038306,0.190149,-0.051429,0.176302,120.678545
